# AION-C · MoSE — Entrenamiento Real  *(v2 · PyG Batching)*

Importa el código real desde el ZIP empaquetado por `setup_drive.py`.
**Cero re-implementaciones**: usa `MoSEPipeline`, crystallizer real, CRE con
scratch pad y GRU, decoder con doble cross-attention.

| Fase | Steps | Datos |
|------|-------|-------|
| Shared pretraining | 1 000 | 5 generadores mezclados |
| CORA fine-tune | 2 000 | `CausalGraphGenerator` |
| FORGE-C fine-tune | 2 000 | `CodeGraphGenerator` |
| MUSE fine-tune | 2 000 | `NarrativeGraphGenerator` |
| EMPATHY fine-tune | 2 000 | `SocialGraphGenerator` |
| AXIOM fine-tune | 500 | `MathGraphGenerator` |

Config: `MoSEConfig.tiny()` — hidden_dim=64, vocab_size=512, ~2M parámetros.

**Optimizaciones activas (Plan v4 §16.2):**
- `PyGStyleBatcher`: concatena B grafos en super-grafo → 1 forward pass del CRE.
- `AutoScaler`: detecta `batch_size` máximo para el hardware disponible.
- `PrecomputedDataset`: pre-tokeniza todos los ejemplos antes del loop.

Tiempo estimado: **< 10 min en T4** (vs ~30 min sin batching).

In [ ]:
# ── 1. Mount Drive · Upload ZIP · Unzip · sys.path ────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile, shutil

DRIVE_ROOT  = '/content/drive/MyDrive'
ZIP_NAME    = 'aion_c.zip'
DRIVE_ZIP   = f'{DRIVE_ROOT}/{ZIP_NAME}'
CONTENT_DIR = '/content/aion_c'

if not os.path.exists(DRIVE_ZIP):
    print(f'ZIP no encontrado en {DRIVE_ZIP}')
    print('Sube aion_c.zip generado por setup_drive.py:')
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        with open(DRIVE_ZIP, 'wb') as f:
            f.write(data)
        print(f'Guardado en {DRIVE_ZIP} ({len(data)/1024:.0f} KB)')
else:
    sz = os.path.getsize(DRIVE_ZIP) / 1024
    print(f'ZIP encontrado: {DRIVE_ZIP} ({sz:.0f} KB)')

if os.path.exists(CONTENT_DIR):
    shutil.rmtree(CONTENT_DIR)
os.makedirs(CONTENT_DIR)

with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall(CONTENT_DIR)
    n = len(zf.namelist())

if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f'Extraidos {n} archivos en {CONTENT_DIR}')
print(f'sys.path[0] = {sys.path[0]}')

In [ ]:
# ── 2. GPU Check · Imports Reales ──────────────────────────────────────────

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU — Runtime > Cambiar tipo de entorno > T4 GPU')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
_f, _t = torch.cuda.mem_get_info()
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {_f/1e9:.1f} / {_t/1e9:.1f} GB libre')

# Pipeline completo
from router.pipeline import MoSEPipeline, MoSEConfig, MoSEOutput

# Motores individuales (para inspección)
from motors.cora.motor    import CORAMotor
from motors.forge_c.motor import CodeMotor
from motors.muse.motor    import CreativeMotor
from motors.axiom.motor   import MathMotor
from motors.empathy.motor import SocialMotor

# Generadores sintéticos
from synth.causal_graph_gen    import CausalGraphGenerator
from synth.code_graph_gen      import CodeGraphGenerator
from synth.math_graph_gen      import MathGraphGenerator
from synth.narrative_graph_gen import NarrativeGraphGenerator
from synth.social_graph_gen    import SocialGraphGenerator

# Metadatos del orchestrator
from orchestrator.model import KEYWORD_TRIGGERS, MOTOR_NAMES

# ── PyG Batching (Plan v4 §16.2) ─────────────────────────────────────────
from cre import PyGStyleBatcher, AutoScaler, AutoScaleResult

print('Todos los imports OK')
print(f'Motores: {MOTOR_NAMES}')

In [ ]:
# ── 3. Config · AutoScaler · Tokenizer · PrecomputedDataset ────────────────

import random, math, time
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

torch.manual_seed(42); random.seed(42)

# Config tiny: hidden_dim=64, vocab_size=512, max_seq_len=128
cfg = MoSEConfig.tiny()
print(f'Config: hidden_dim={cfg.hidden_dim}, vocab_size={cfg.vocab_size}, '
      f'max_seq_len={cfg.dec_max_seq_len}, K={cfg.motor_max_nodes}')

# ── Tokenizador char-level ────────────────────────────────────────────────
PAD, BOS, EOS = 0, 1, 2

def encode(text: str, max_len: int = None) -> list:
    ids = [BOS] + [(ord(c) % (cfg.vocab_size - 3)) + 3 for c in text] + [EOS]
    if max_len:
        ids = ids[:max_len]
    return ids

def decode(ids: list) -> str:
    out = []
    for i in ids:
        if i == EOS:
            break
        if i > 2:
            out.append(chr((i - 3) % (cfg.vocab_size - 3)))
    return ''.join(out)

def make_batch(texts, device, max_len=None):
    L = max_len or cfg.dec_max_seq_len
    seqs = [encode(t, max_len=L) for t in texts]
    maxl = max(len(s) for s in seqs)
    padded = [s + [PAD] * (maxl - len(s)) for s in seqs]
    return torch.tensor(padded, dtype=torch.long, device=device)

# ── Generadores por dominio ───────────────────────────────────────────────
GENS = {
    'cora':    CausalGraphGenerator(),
    'forge_c': CodeGraphGenerator(),
    'axiom':   MathGraphGenerator(),
    'muse':    NarrativeGraphGenerator(),
    'empathy': SocialGraphGenerator(),
}

MOTOR_HINTS = {
    'cora':    '',
    'forge_c': 'function ',
    'axiom':   'theorem ',
    'muse':    'historia ',
    'empathy': 'siente ',
}

# ── PrecomputedDataset: pre-tokeniza Y pre-computa grafos ────────────────
class PrecomputedDataset:
    """
    Pre-genera, pre-tokeniza y (opcionalmente) pre-computa grafos del
    crystallizer para todos los N ejemplos de un dominio.
    Elimina crystallizer + generate()+encode() del inner loop.
    Tokens en CPU; node_vecs en CPU (se mueven a GPU en get_batch).
    """
    def __init__(self, domain: str, n: int, device, levels=(1, 2)):
        gen  = GENS[domain]
        hint = MOTOR_HINTS[domain]
        ids_list = []
        for _ in range(n):
            lvl = random.choice(levels)
            ex  = gen.generate(level=lvl)
            text = ex.problem_text + ' ' + ex.answer
            ids_list.append(encode(text, max_len=cfg.dec_max_seq_len))
        self._ids         = ids_list
        self.hint         = hint
        self.domain       = domain
        self.device       = device
        self.n            = len(ids_list)
        self._graphs      = None   # List[CausalGraph]
        self._node_vecs   = None   # List[Tensor cpu [n_i, D]]
        self._node_counts = None   # List[int]

    def precompute_graphs(self, pipeline, motor_name: str,
                          batch_size: int = 8) -> None:
        """Corre el crystallizer UNA VEZ para todos los ejemplos y guarda
        los grafos + node_vectors en CPU. Después de llamar esto, get_batch
        devuelve los grafos pre-computados y el crystallizer no se ejecuta
        en el training loop."""
        motor = pipeline.motors[motor_name]
        graphs = []; nvecs = []; ncounts = []
        pipeline.eval()
        with torch.no_grad():
            for start in range(0, self.n, batch_size):
                chunk  = self._ids[start:start + batch_size]
                maxl   = max(len(s) for s in chunk)
                padded = [s + [PAD] * (maxl - len(s)) for s in chunk]
                ids_t  = torch.tensor(padded, dtype=torch.long, device=self.device)
                concepts = pipeline.encoder(ids_t)
                cryst    = motor.build_graph(concepts)
                for b in range(len(chunk)):
                    n = cryst.node_counts[b]
                    graphs.append(cryst.graphs[b])
                    nvecs.append(cryst.node_vectors[b, :n].cpu())
                    ncounts.append(n)
        self._graphs      = graphs
        self._node_vecs   = nvecs
        self._node_counts = ncounts
        pipeline.train()

    def get_batch(self, bs: int):
        """Muestra bs ejemplos. Retorna (token_ids, graph_data|None).
        graph_data = (graphs, node_vecs_gpu, ncounts) si precompute_graphs
        fue llamado; None si no (fallback al crystallizer en el forward)."""
        indices = random.choices(range(self.n), k=bs)
        seqs    = [self._ids[i] for i in indices]
        maxl    = max(len(s) for s in seqs)
        padded  = [s + [PAD] * (maxl - len(s)) for s in seqs]
        token_ids = torch.tensor(padded, dtype=torch.long, device=self.device)
        if self._graphs is None:
            return token_ids, None
        graphs    = [self._graphs[i]                      for i in indices]
        node_vecs = [self._node_vecs[i].to(self.device)   for i in indices]
        ncounts   = [self._node_counts[i]                 for i in indices]
        return token_ids, (graphs, node_vecs, ncounts)

# ── Pipeline ──────────────────────────────────────────────────────────────
pipeline = MoSEPipeline(cfg).to(DEVICE)

# Gradient checkpointing: recomputa activaciones en el backward en lugar de
# guardarlas — reduce VRAM del backward ~2-3× a costa de ~15% más compute.
# Crítico cuando backward=67% del tiempo: libera VRAM para batch más grandes.
pipeline.encoder.enable_gradient_checkpointing()
pipeline.decoder.enable_gradient_checkpointing()
print('Gradient checkpointing: encoder OK, decoder OK')

bd = pipeline.parameter_breakdown()
print()
print(f"  {'Módulo':<22} {'Parámetros':>12}")
print(f"  {'-'*36}")
for k, v in bd.items():
    if k != 'total_unique':
        print(f"  {k:<22} {v:>12,}")
print(f"  {'-'*36}")
print(f"  {'TOTAL (unique)':<22} {bd['total_unique']:>12,}")

# ── AutoScaler: detectar batch_size óptimo ────────────────────────────────
# Usamos el CRE del motor CORA como proxy (mismo hidden_dim para todos)
from core.graph import CausalEdge, CausalGraph, CausalNode, CausalRelation, NodeType

def _make_sample_graph(n=8, e=6):
    g = CausalGraph()
    for i in range(n):
        g.add_node(CausalNode(f'n{i}', f'n{i}', NodeType.EVENT, 0.9))
    for i in range(min(e, n-1)):
        g.add_edge(CausalEdge(f'n{i}', f'n{i+1}',
                              CausalRelation.CAUSES, 0.8, 0.9))
    return g

_sample_graph = _make_sample_graph()
_scaler       = AutoScaler()
_cre_proxy    = pipeline.motors['cora'].cre

_scale_result = _scaler.find_optimal_batch(
    _cre_proxy, _sample_graph, DEVICE,
    node_dim     = cfg.hidden_dim,
    n_iterations = 3,
)
print()
_scale_result.print_summary()
BATCH = max(min(_scale_result.batch_size, 8), 4)   # cap=8: sweet spot con backward+checkpointing
SAMP_S = _scale_result.samples_per_sec            # throughput CRE medido

# Info de hardware
_f2, _t2 = torch.cuda.mem_get_info()
_used_gb  = (_t2 - _f2) / 1e9
print()
print(f'AutoScaler → batch_size = {BATCH}')
print(f'VRAM usada : {_used_gb:.2f} GB / {_t2/1e9:.1f} GB '
      f'({100*(_t2-_f2)/_t2:.1f}%)')

# ── Pre-generar datasets (elimina generate() del inner loop) ─────────────
_N_EXAMPLES = 512   # ejemplos por dominio
print()
print(f'Pre-generando {_N_EXAMPLES} ejemplos por dominio...', end=' ', flush=True)
_t0_pre = time.perf_counter()

DATASETS = {
    d: PrecomputedDataset(d, _N_EXAMPLES, DEVICE)
    for d in MOTOR_NAMES
}

print(f'OK ({time.perf_counter()-_t0_pre:.1f}s)')
print('Datasets listos:', {d: ds.n for d, ds in DATASETS.items()})

# ── Pre-computar grafos del crystallizer para todos los dominios ──────────
# Ejecuta crystallizer UNA VEZ por dominio sobre todos los ejemplos.
# Esto elimina crystallizer del forward loop Y sus Slice/Select/IndexBackward
# del backward — esas ops eran 43% del CUDA time total.
print()
print(f'Pre-computando grafos ({_N_EXAMPLES} ej × {len(MOTOR_NAMES)} dominios)...',
      end=' ', flush=True)
_t0_gc = time.perf_counter()
for _d in MOTOR_NAMES:
    DATASETS[_d].precompute_graphs(pipeline, _d)
print(f'OK ({time.perf_counter()-_t0_gc:.1f}s) — crystallizer eliminado del loop')

pipeline.train()

In [ ]:
# ── 4. Helpers · PyGStyleBatcher · batched_pipeline_forward ────────────────
#
# Reemplaza el loop `for b in range(B): motor.reason(graph_b, feats_b)`
# con UN SOLO `motor.cre.forward_batched(batched_B_grafos)`.
# Matemáticamente idéntico. ~13x más rápido con batch=16 (Plan v4 §16.2.3).

# ── Helpers de progreso ──────────────────────────────────────────────────
def _pbar(done: int, total: int, w: int = 22) -> str:
    """Barra de progreso ASCII: [████████░░░░] 64%"""
    frac   = done / max(total, 1)
    filled = int(frac * w)
    bar    = '█' * filled + '░' * (w - filled)
    return f'[{bar}] {frac*100:>5.1f}%'

def _fmt_eta(secs: float) -> str:
    """Formatea segundos como mm:ss o h:mm:ss."""
    secs = int(max(0, secs))
    h, r = divmod(secs, 3600)
    m, s = divmod(r, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

_LOG_INTERVAL = 30.0   # segundos entre prints de progreso

_batcher = PyGStyleBatcher()

def batched_pipeline_forward(
    pipe,
    token_ids:   torch.Tensor,
    query_text:  str = None,
    precomp:     dict = None,
) -> MoSEOutput:
    """
    Forward pass del MoSEPipeline con CRE batching estilo PyG.

    Diferencia con pipe.forward():
        Antiguo: for b in range(B): motor.reason(graph_b, feats_b)  ← B llamadas
        Nuevo:   motor.cre.forward_batched(batch_de_B_grafos)        ← 1 llamada

    Resultado idéntico. El scatter_add con edge_index offseteados garantiza
    que los grafos no se mezclan (ver cre/batching.py).
    """
    B, L   = token_ids.shape
    D      = pipe.config.hidden_dim
    K      = pipe.config.motor_max_nodes
    device = token_ids.device
    dtype  = pipe.encoder.token_embedding.weight.dtype

    # ── 1. Encode ──────────────────────────────────────────────────────────
    with torch.autograd.profiler.record_function('ENCODER_forward'):
        concepts = pipe.encoder(token_ids)       # [B, L, D]

    # ── 2. Orchestrate ─────────────────────────────────────────────────────
    orch_out = pipe.orchestrator(concepts, query_text)

    # ── 3. Crystallizer (o pre-computado si disponible) ───────────────────
    # precomp: {'motor': str, 'graphs': [...], 'node_vecs': [...], 'ncounts': [...]}
    # Si disponible para el motor activo, salta el crystallizer por completo
    # — elimina Slice/Select/IndexBackward del grafo computacional.
    # Si no, corre crystallizer con no_grad (fallback seguro).
    motor_cryst = {}
    for act in orch_out.activations:
        if precomp is not None and act.motor_name == precomp['motor']:
            continue   # se usa precomp directamente abajo
        with torch.no_grad():
            motor_cryst[act.motor_name] = pipe.motors[act.motor_name].build_graph(concepts)

    # ── 4. CRE batching: 1 forward_batched por motor (en vez de B llamadas) ─
    motor_cre_outs = {}   # motor_name -> List[CREOutput | None] de longitud B

    for act in orch_out.activations:
        motor    = pipe.motors[act.motor_name]
        graphs_b = []; node_feats_b = []; valid_b = []

        if precomp is not None and act.motor_name == precomp['motor']:
            # Ruta rápida: grafos pre-computados, sin crystallizer
            pg = precomp['graphs']
            pv = precomp['node_vecs']    # ya en GPU (movidos en get_batch)
            pn = precomp['ncounts']
            for b in range(B):
                if pn[b] > 0:
                    graphs_b.append(pg[b])
                    # node_vecs pre-computados: detach del crystallizer histórico,
                    # requires_grad=True para que CRE+decoder reciban gradientes.
                    node_feats_b.append(pv[b].detach().requires_grad_(True))
                    valid_b.append(b)
        else:
            cryst_out = motor_cryst[act.motor_name]
            for b in range(B):
                n = cryst_out.node_counts[b]
                if n > 0:
                    graphs_b.append(cryst_out.graphs[b])
                    node_feats_b.append(
                        cryst_out.node_vectors[b, :n].detach().requires_grad_(True))
                    valid_b.append(b)

        if not graphs_b:
            motor_cre_outs[act.motor_name] = [None] * B
            continue

        # UN SOLO forward pass para todos los grafos válidos del batch
        batched  = _batcher.batch(graphs_b, node_feats_b)
        with torch.autograd.profiler.record_function('CRE_forward'):
            cre_outs = motor.cre.forward_batched(
                batched, n_iterations=act.n_iterations
            )

        # Mapear resultados de vuelta a posiciones B
        cre_per_b = [None] * B
        for i, b in enumerate(valid_b):
            cre_per_b[b] = cre_outs[i]
        motor_cre_outs[act.motor_name] = cre_per_b

    # ── 5. Build graph_repr [B, K, D] ──────────────────────────────────────
    # Para cada item del batch: get_graph_repr() + unifier
    all_graph_reprs = []
    last_unif_out   = None

    for b in range(B):
        motor_reprs = []
        for act in orch_out.activations:
            motor   = pipe.motors[act.motor_name]
            cre_out = motor_cre_outs[act.motor_name][b]
            if cre_out is None:
                motor_reprs.append(torch.zeros(K, D, device=device, dtype=dtype))
            else:
                motor_reprs.append(motor.get_graph_repr(cre_out, k_nodes=K))

        last_unif_out = pipe.unifier(motor_reprs)
        all_graph_reprs.append(last_unif_out.unified)   # [K, D]

    graph_repr = torch.stack(all_graph_reprs, dim=0)    # [B, K, D]

    # ── 6. Decode ──────────────────────────────────────────────────────────
    with torch.autograd.profiler.record_function('DECODER_forward'):
        dec_out = pipe.decoder(token_ids, graph_repr, concepts)

    return MoSEOutput(
        logits        = dec_out.logits,
        anchor_logits = dec_out.anchor_logits,
        confidence    = dec_out.confidence,
        needs_clarif  = dec_out.needs_clarification,
        graph_repr    = graph_repr,
        orchestrator  = orch_out,
        unifier       = last_unif_out,
        active_motors = orch_out.motor_names,
    )

print('batched_pipeline_forward listo')
print(f'Usando batch_size={BATCH} | PyGStyleBatcher activo')

# Smoke test: 1 step para verificar shapes
pipeline.eval()
with torch.no_grad(), autocast():
    _ids_test, _gd_test = DATASETS['cora'].get_batch(2)
    _pc_test = ({'motor': 'cora', 'graphs': _gd_test[0],
                 'node_vecs': _gd_test[1], 'ncounts': _gd_test[2]}
                if _gd_test is not None else None)
    _out_test = batched_pipeline_forward(pipeline, _ids_test, 'cora', precomp=_pc_test)
print(f'Smoke test OK: logits {_out_test.logits.shape}, '
      f'active_motors={_out_test.active_motors}')

# ── Pipeline timing probe: forward + backward completo (realista) ────────
# Mide el pipeline ENTERO (encoder + crystallizer + CRE + decoder + backward)
# con el mismo batch_size que se usará en training. Así el throughput y los
# estimados de tiempo son realistas, no el throughput del CRE solo.
print('\nPipeline timing probe — forward+backward (3 pasos)...', end=' ', flush=True)
_probe_ids, _probe_gd = DATASETS['cora'].get_batch(BATCH)
_probe_pc = ({'motor': 'cora', 'graphs': _probe_gd[0],
              'node_vecs': _probe_gd[1], 'ncounts': _probe_gd[2]}
             if _probe_gd is not None else None)
_probe_opt = torch.optim.SGD(pipeline.parameters(), lr=1e-4)
pipeline.train()
_pipe_times = []
for _ in range(3):
    torch.cuda.synchronize()              # flush GPU antes de empezar
    _t0p = time.perf_counter()
    with autocast():
        _probe_out  = batched_pipeline_forward(pipeline, _probe_ids, 'cora',
                                               precomp=_probe_pc)
        _probe_loss = F.cross_entropy(
            _probe_out.logits[:, :-1].reshape(-1, cfg.vocab_size),
            _probe_ids[:, 1:].reshape(-1),
            ignore_index=PAD)
    _probe_loss.backward()
    _probe_opt.step()
    _probe_opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()              # esperar a que GPU termine
    _pipe_times.append(time.perf_counter() - _t0p)
del _probe_opt
_avg_step_s = sum(_pipe_times) / len(_pipe_times)
SAMP_S_PIPE = BATCH / _avg_step_s
print(f'OK | {_avg_step_s*1000:.0f}ms/step | {SAMP_S_PIPE:.0f} samp/s')

# ── Breakdown de timing por componente (3 pasos) ─────────────────────────
print('Component breakdown (3 pasos)...', end=' ', flush=True)
_B   = BATCH
_D   = pipeline.config.hidden_dim
_K   = pipeline.config.motor_max_nodes
_dev = _probe_ids.device
_dty = pipeline.encoder.token_embedding.weight.dtype
_bopt = torch.optim.SGD(pipeline.parameters(), lr=1e-4)
_t_enc = _t_cryst = _t_cre = _t_dec = _t_bwd = 0.0

for _pi in range(3):
    # Encoder + orchestrator
    torch.cuda.synchronize(); _t0 = time.perf_counter()
    with autocast():
        _pc = pipeline.encoder(_probe_ids)
        _po = pipeline.orchestrator(_pc, 'cora')
    torch.cuda.synchronize(); _t_enc += time.perf_counter() - _t0

    # Crystallizer
    torch.cuda.synchronize(); _t0 = time.perf_counter()
    _pmc = {}
    with autocast():
        for _act in _po.activations:
            _pmc[_act.motor_name] = pipeline.motors[_act.motor_name].build_graph(_pc)
    torch.cuda.synchronize(); _t_cryst += time.perf_counter() - _t0

    # CRE (forward_batched)
    torch.cuda.synchronize(); _t0 = time.perf_counter()
    _pco = {}
    with autocast():
        for _act in _po.activations:
            _mot = pipeline.motors[_act.motor_name]; _cx = _pmc[_act.motor_name]
            _gb = []; _nf = []; _vb = []
            for _b in range(_B):
                _n = _cx.node_counts[_b]
                if _n > 0:
                    _gb.append(_cx.graphs[_b])
                    _nf.append(_cx.node_vectors[_b, :_n])
                    _vb.append(_b)
            if _gb:
                _bat = _batcher.batch(_gb, _nf)
                _cr  = _mot.cre.forward_batched(_bat, n_iterations=_act.n_iterations)
                _cp  = [None] * _B
                for _i, _b in enumerate(_vb): _cp[_b] = _cr[_i]
                _pco[_act.motor_name] = _cp
            else:
                _pco[_act.motor_name] = [None] * _B
    torch.cuda.synchronize(); _t_cre += time.perf_counter() - _t0

    # Decoder (get_graph_repr + unifier + decoder)
    torch.cuda.synchronize(); _t0 = time.perf_counter()
    with autocast():
        _pgr = []
        for _b in range(_B):
            _mr = []
            for _act in _po.activations:
                _mot = pipeline.motors[_act.motor_name]
                _co  = _pco[_act.motor_name][_b]
                _mr.append(_mot.get_graph_repr(_co, k_nodes=_K) if _co is not None
                           else torch.zeros(_K, _D, device=_dev, dtype=_dty))
            _pgr.append(pipeline.unifier(_mr).unified)
        _gr  = torch.stack(_pgr, 0)
        _do  = pipeline.decoder(_probe_ids, _gr, _pc)
        _pl  = F.cross_entropy(_do.logits[:, :-1].reshape(-1, cfg.vocab_size),
                               _probe_ids[:, 1:].reshape(-1), ignore_index=PAD)
    torch.cuda.synchronize(); _t_dec += time.perf_counter() - _t0

    # Backward
    torch.cuda.synchronize(); _t0 = time.perf_counter()
    _pl.backward()
    torch.cuda.synchronize(); _t_bwd += time.perf_counter() - _t0
    _bopt.step(); _bopt.zero_grad(set_to_none=True)

del _bopt
_ts = [_t_enc/3, _t_cryst/3, _t_cre/3, _t_dec/3, _t_bwd/3]
_tn = ['encoder+orch', 'crystallizer', 'CRE (batched)', 'decoder+unif', 'backward']
_tt = sum(_ts)
print('OK')
print()
print(f"  {'Component':<18} {'ms/step':>8} {'% total':>8}")
print(f"  {'-'*36}")
for _n, _t in zip(_tn, _ts):
    print(f"  {_n:<18} {_t*1000:>7.1f}  {100*_t/_tt:>7.1f}%")
print(f"  {'-'*36}")
print(f"  {'TOTAL':<18} {_tt*1000:>7.1f}  {'100.0%':>8}")

# ── Autograd profiler: top 20 ops por CUDA time (forward + backward) ────
print('\nAutograd profiler (1 step, use_cuda=True)...', end=' ', flush=True)
_popt = torch.optim.SGD(pipeline.parameters(), lr=1e-4)
pipeline.train()
with torch.autograd.profiler.profile(use_cuda=True, record_shapes=False) as _prof:
    with autocast():
        _prof_out  = batched_pipeline_forward(pipeline, _probe_ids, 'cora',
                                              precomp=_probe_pc)
        with torch.autograd.profiler.record_function('LOSS'):
            _prof_loss = F.cross_entropy(
                _prof_out.logits[:, :-1].reshape(-1, cfg.vocab_size),
                _probe_ids[:, 1:].reshape(-1),
                ignore_index=PAD)
    _prof_loss.backward()
    _popt.step()
    _popt.zero_grad(set_to_none=True)
del _popt
print('OK')
print()
print('  Top 20 operaciones por cuda_time_total (forward + backward):')
print(_prof.key_averages().table(sort_by='cuda_time_total', row_limit=20))

# Estimados de tiempo total
_STEPS_TOTAL = 1000 + sum({'cora':2000,'forge_c':2000,'axiom':500,'muse':2000,'empathy':2000}.values())
_eta_total   = _STEPS_TOTAL * _avg_step_s
_eta_ph1     = 1000 * _avg_step_s
print()
print(f'  batch_size     : {BATCH}  (AutoScaler timing probe)')
print(f'  CRE samp/s     : {SAMP_S:.0f}')
print(f'  Pipeline samp/s: {SAMP_S_PIPE:.0f}')
print(f'  Estimado Phase 1 (1000 steps): {_fmt_eta(_eta_ph1)}')
print(f'  Estimado total  ({_STEPS_TOTAL} steps): {_fmt_eta(_eta_total)}')
pipeline.train()

In [ ]:
# ── 5. Phase 1: Shared Pretraining (con PyG batching) ──────────────────────
# 1000 steps con datos mezclados de los 5 dominios.
# Cada step: forward_batched (CRE en 1 pass) vs forward (CRE en B passes).

LR    = 3e-4
STEPS = 1_000

scaler = GradScaler()
opt    = torch.optim.AdamW(pipeline.parameters(), lr=LR, weight_decay=0.01)
sched  = torch.optim.lr_scheduler.OneCycleLR(
             opt, LR, total_steps=STEPS, pct_start=0.1, anneal_strategy='cos')

t_total = time.perf_counter()
losses_shared = []
pipeline.train()

# Generador de batches mezclados — usa datasets + grafos pre-computados
def get_mixed_precomputed():
    domain = random.choice(MOTOR_NAMES)
    ids, gd = DATASETS[domain].get_batch(BATCH)
    pc = ({'motor': domain, 'graphs': gd[0], 'node_vecs': gd[1], 'ncounts': gd[2]}
          if gd is not None else None)
    return ids, MOTOR_HINTS[domain], pc

_last_log_ph1 = time.perf_counter()

for step in range(1, STEPS + 1):
    ids, hint, precomp_s = get_mixed_precomputed()

    with autocast():
        out  = batched_pipeline_forward(pipeline, ids, hint or None,
                                        precomp=precomp_s)
        loss = F.cross_entropy(
            out.logits[:, :-1].reshape(-1, cfg.vocab_size),
            ids[:, 1:].reshape(-1),
            ignore_index=PAD)

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(pipeline.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    opt.zero_grad(set_to_none=True)
    sched.step()
    losses_shared.append(loss.item())

    _now = time.perf_counter()
    if _now - _last_log_ph1 >= _LOG_INTERVAL:
        _last_log_ph1 = _now
        avg = sum(losses_shared[-50:]) / min(50, len(losses_shared))
        lr  = sched.get_last_lr()[0]
        t_e = _now - t_total
        thr = step * BATCH / t_e
        eta = (STEPS - step) * (t_e / step)
        print(f'  {_pbar(step, STEPS)} step {step:>5}/{STEPS}  '
              f'loss={avg:.4f}  lr={lr:.2e}  {thr:.0f} samp/s  ETA {_fmt_eta(eta)}')

t_ph1 = time.perf_counter() - t_total
p0 = sum(losses_shared[:10]) / 10
p1 = sum(losses_shared[-10:]) / 10
print(f'\nPhase 1 done | loss {p0:.4f} → {p1:.4f} | {t_ph1:.0f}s '
      f'({t_ph1/60:.1f} min) | {STEPS*BATCH/t_ph1:.0f} samples/s')

In [ ]:
# ── 6. Phase 2: Per-Motor Fine-Tuning (con PyG batching) ───────────────────

LR_MOTOR  = 1e-4
LR_SHARED = 2e-5

STEPS_PER_MOTOR = {
    'cora':    2000,
    'forge_c': 2000,
    'axiom':   500,
    'muse':    2000,
    'empathy': 2000,
}

# ── Métricas ─────────────────────────────────────────────────────────────
def word_f1(pred: str, ref: str) -> float:
    p, r = set(pred.lower().split()), set(ref.lower().split())
    if not p or not r: return 0.0
    tp = len(p & r)
    pr = tp / len(p); rc = tp / len(r)
    return round(2 * pr * rc / (pr + rc), 3) if (pr + rc) else 0.0

@torch.no_grad()
def greedy_decode(prompt: str, hint: str, max_new: int = 48) -> str:
    pipeline.eval()
    ids = encode(prompt, max_len=cfg.dec_max_seq_len - max_new)
    cur = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new):
        if cur.shape[1] >= cfg.dec_max_seq_len: break
        out = pipeline(cur, query_text=hint or None)
        nxt = out.logits[0, -1].argmax().item()
        if nxt == EOS: break
        cur = torch.cat([cur, torch.tensor([[nxt]], device=DEVICE)], 1)
    pipeline.train()
    return decode(cur[0, len(ids):].tolist())

def eval_motor(domain: str, n: int = 3) -> float:
    hint = MOTOR_HINTS[domain]
    gen  = GENS[domain]
    f1s  = []
    print(f'  Eval [{domain}]:')
    for i in range(n):
        ex = gen.generate(level=random.choice([1, 2]))
        q  = ex.problem_text
        a  = ex.answer
        p  = greedy_decode(q, hint, max_new=len(a) + 8)
        f1 = word_f1(p, a)
        f1s.append(f1)
        print(f'    [{i+1}] Q: {q[:58]}{"..." if len(q)>58 else ""}')
        print(f'         A: {a[:58]}')
        print(f'         P: {p[:58]}{"..." if len(p)>58 else ""}  F1={f1:.2f}')
    mean = sum(f1s) / len(f1s) if f1s else 0.0
    print(f'         → mean Word F1 = {mean:.3f}')
    return mean

# ── Training loop por motor ───────────────────────────────────────────────
orig_conf = pipeline.orchestrator.config.min_confidence_to_activate
pipeline.orchestrator.config.min_confidence_to_activate = 1.0

motor_results = {}
SEP = '━'

for domain in MOTOR_NAMES:
    steps_m = STEPS_PER_MOTOR[domain]
    print(f'\n{SEP*60}')
    print(f'  Motor {domain.upper():<10}  [{steps_m} steps | data={domain}]')
    print(f'{SEP*60}')

    motor_params  = list(pipeline.motors[domain].parameters())
    shared_params = (list(pipeline.encoder.parameters()) +
                     list(pipeline.decoder.parameters()) +
                     list(pipeline.orchestrator.parameters()) +
                     list(pipeline.unifier.parameters()))
    opt_m = torch.optim.AdamW(
        [{'params': motor_params,  'lr': LR_MOTOR},
         {'params': shared_params, 'lr': LR_SHARED}],
        weight_decay=0.01)

    warmup = min(100, steps_m // 10)
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, steps_m - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    sched_m = torch.optim.lr_scheduler.LambdaLR(opt_m, lr_lambda)

    losses_m = []; t_m = time.perf_counter()
    pipeline.train()

    # Dataset pre-computado para este motor
    ds_m = DATASETS[domain]
    _last_log_m = time.perf_counter()

    for step in range(1, steps_m + 1):
        ids, _gd_m = ds_m.get_batch(BATCH)
        hint       = ds_m.hint
        precomp_m  = ({'motor': domain, 'graphs': _gd_m[0],
                       'node_vecs': _gd_m[1], 'ncounts': _gd_m[2]}
                      if _gd_m is not None else None)

        with autocast():
            out  = batched_pipeline_forward(pipeline, ids, hint or None,
                                            precomp=precomp_m)
            loss = F.cross_entropy(
                out.logits[:, :-1].reshape(-1, cfg.vocab_size),
                ids[:, 1:].reshape(-1),
                ignore_index=PAD)

        scaler.scale(loss).backward()
        scaler.unscale_(opt_m)
        nn.utils.clip_grad_norm_(pipeline.parameters(), 1.0)
        scaler.step(opt_m); scaler.update()
        opt_m.zero_grad(set_to_none=True)
        sched_m.step()
        losses_m.append(loss.item())

        _now = time.perf_counter()
        if _now - _last_log_m >= _LOG_INTERVAL:
            _last_log_m = _now
            avg = sum(losses_m[-50:]) / min(50, len(losses_m))
            t_e = _now - t_m
            thr = step * BATCH / t_e
            eta = (steps_m - step) * (t_e / step)
            print(f'    {_pbar(step, steps_m)} step {step:>4}/{steps_m}  '
                  f'loss={avg:.4f}  {thr:.0f} samp/s  ETA {_fmt_eta(eta)}')

    f1 = eval_motor(domain)
    l0 = losses_m[0]; l1 = sum(losses_m[-50:]) / min(50, len(losses_m))
    motor_results[domain] = {'loss_start': l0, 'loss_end': l1, 'f1': f1,
                              'time_s': time.perf_counter() - t_m}

pipeline.orchestrator.config.min_confidence_to_activate = orig_conf

In [ ]:
# ── 7. Resumen Final ────────────────────────────────────────────────────────

SEP = '━'
W   = 68
print(f'\n{SEP*W}')
print('  MoSE Real v2 — Resumen de Entrenamiento (PyG Batching)')
print(f'{SEP*W}')

# Hardware
_f3, _t3 = torch.cuda.mem_get_info()
print(f'  GPU            : {torch.cuda.get_device_name(0)}')
print(f'  batch_size     : {BATCH} (AutoScaler)')
print(f'  VRAM usada     : {(_t3-_f3)/1e9:.2f} GB / {_t3/1e9:.1f} GB'
      f' ({100*(_t3-_f3)/_t3:.1f}%)')
print()

# Shared pretraining
p0 = sum(losses_shared[:10]) / 10
p1 = sum(losses_shared[-10:]) / 10
print(f'  Shared pretraining : {p0:.4f} → {p1:.4f}  ({t_ph1:.0f}s)')
print()

# Per-motor
print(f"  {'Motor':<12} {'Steps':>6} {'Loss0':>8} {'LossF':>8} "
      f"{'Word F1':>9} {'Time':>7} {'Samp/s':>8}")
print(f"  {'-'*62}")
for d, r in motor_results.items():
    s   = STEPS_PER_MOTOR[d]
    thr = s * BATCH / r['time_s'] if r['time_s'] > 0 else 0
    print(f"  {d:<12} {s:>6} {r['loss_start']:>8.4f} {r['loss_end']:>8.4f} "
          f"{r['f1']:>9.3f} {r['time_s']:>6.0f}s {thr:>8.0f}")
print()

# Routing check
print('  Routing (heurístico, min_confidence restaurado):')
ROUTING_QUERIES = {
    'cora':    'rain causes flood. does rain cause flood?',
    'forge_c': 'function getData calls parseJSON. if parseJSON breaks?',
    'axiom':   'a>b and b>c. theorem: is a>c?',
    'muse':    'el héroe de la historia quiere libertad',
    'empathy': 'Ana siente que Pedro está enojado pero no lo está',
}
pipeline.eval()
with torch.no_grad():
    for expected, q in ROUTING_QUERIES.items():
        ids = make_batch([q], DEVICE)
        out = pipeline(ids, query_text=q)
        got  = out.active_motors[0]
        mark = 'OK' if got == expected else f'FAIL->{got}'
        print(f'    {expected:<12} {mark}')
print()

# Totales
total_s = time.perf_counter() - t_total
print(f'  Params totales : {pipeline.count_parameters():,}')
print(f'  Tiempo total   : {total_s:.0f}s ({total_s/60:.1f} min)')
print(f'  Throughput med : {(STEPS + sum(STEPS_PER_MOTOR.values())) * BATCH / total_s:.0f} samples/s')
print(f'  Scale-up       : cfg = MoSEConfig(hidden_dim=512, ...) — sin cambios de código')
print(f'{SEP*W}')